# 每日選股流程（重構版）

**用法**：Kernel → Restart & Run All，由上而下執行即可。要補跑某一天，改第 1 格的 `TARGET_DATE_OVERRIDE`。

| 區塊 | 內容 | 以後要改什麼就來這裡 |
|---|---|---|
| 1. 設定 | 路徑、日期、名單、門檻、輸出欄位 | 持股/觀察名單、門檻、換資料夾 |
| 2. 工具 | 數值轉換 `to_float` / `to_num` / `num`、百分比欄位表 `PCT_SPECS` | 數值轉換一律走這裡；新增乖離率/比值欄位 → `PCT_SPECS` 加一行 |
| 3. 日期 | 推算 `adate` / `targetday` / `date_list` | 日期邏輯 |
| 4. 載入 | 讀每日檔、即時檔、基本資料、供應鏈 | 資料來源 |
| 5. 特徵 | 清洗＋計算指標（**全程維持數值**） | 新增指標 |
| 6. 統計 | Typelevel / 族群成交金額、強勢股數 | 統計口徑 |
| 7. 規則 | `RULES` 表：標籤＋條件＋輸出資料夾 | 新增/停用選股條件 → 改 `RULES` |
| 8. 匯出 | 只在匯出時才轉成字串；匯出欄位 = `OUTPUT_COLS` + `SECTOR_COLS`（族群、子族群） | 輸出格式、欄位 |
| 9. 族群檔 / summary.json | 依族群輸出、儀表板用 JSON | |

### 與舊版行為不同的地方（請先對照一次輸出）
1. **`營業毛利率`、`營業利益率`**：舊版漏了 `.str`，`%` 其實沒被去掉；現在會去掉（與 `殖利率` 一致）。
2. **`_quote > 6`**：舊版是跟字串 `'6'` 比（字串比較），現在是數值比較；只有剛好 6.00 或 ≥10 的極端值會有差異。
3. **日期**：不再有 `2026-01-01` 起算的下限，單純取最近 `N_DAYS` 個日曆日。
4. **`VPC_MA_%`**：舊版有 `df['']` 的筆誤，所以此欄一直是「沒乘 100 的比值」。為了不動到下游，預設維持舊值；要改成百分比把 `VPC_MA_PCT_AS_PERCENT = True`。
5. 不再留下空名稱欄位 `''`；`df` 不會再被 `for df, cols in ...` 覆蓋；index 在各階段 `reset_index`，不再有重複 index。
6. **三大法人欄位**：改依 `(stock_number, 日期)` 併入「當天」的 `fund_result_YYYYMMDD.xlsx`；舊版只讀今天的檔再依代號併，過去日期的列會拿到今天的法人資料。對應鍵也從（代號＋公司名稱）改成只用代號。
7. **checkall**：只寫今天，過去日期已有檔就不重寫（保留當天快照）；要整批重產把 `REBUILD_CHECKALL = True`。沒有資料的日期（週末/假日）不再產生空檔。

### 已知但「刻意保留」的舊行為
- `量倍率`（第 5 節 `_volume_features_one`）：舊版 `pd.Series(ratio)[mask]` 靠 index 對齊，實際上 ratio 與 mask 錯位，我完全照舊算以免數值改變；程式碼裡有註解寫了正確版本的改法。
- `summary.json` 的 `quote` 欄位取的是 `_quote`（當日漲跌幅），不是 `diff_quote`。

## 1. 設定區

In [ ]:
# =============================================================
# 1. 設定區：路徑 / 日期 / 名單 / 門檻 / 輸出欄位，全部集中在這一格
#    以後改持股、改門檻、換資料夾，只需要改這裡，邏輯程式碼不用動
# =============================================================
import os
import re
import json
import warnings
from pathlib import Path
from datetime import datetime, timedelta

import numpy as np
import pandas as pd

warnings.filterwarnings('ignore', category=pd.errors.PerformanceWarning)   # 欄位很多時的碎片化提醒，不影響結果
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 200)

# ---------- 路徑（可用環境變數 STOCK_BASE_DIR 覆蓋） ----------
BASE_DIR          = Path(os.environ.get('STOCK_BASE_DIR', r'D:\Project\Jupyter\Stock\Main'))
RESULT_DIR        = BASE_DIR / 'Result'
DAILY_DIR         = RESULT_DIR / 'DailyFiles'
REALTIME_DIR      = RESULT_DIR / 'DailyRunRealTimeStock'
BASE_INFO_CSV     = BASE_DIR / 'BaseInfo' / 'StockBaseData.csv'
SUPPLY_CHAIN_JSON = BASE_DIR / 'BaseInfo' / 'supply_chain.json'

# ---------- 日期 ----------
TARGET_DATE_OVERRIDE = None   # 例如 '2026-10-13'；None = 沿用舊邏輯自動推算
#TARGET_DATE_OVERRIDE='2026-10-12'
N_DAYS = 15                   # 往回看幾個「日曆日」（含目標日）
REBUILD_CHECKALL = False      # True = 重產 date_list 內所有 checkall；False = 只寫今天＋補缺檔

# ---------- 門檻 ----------
HOT_QUOTE        = 6                     # 族群統計：漲幅 > 6% 算強勢股
LEVEL_UP         = 50                    # Typelevel：LevelArea >= 50 → 右上
LEVEL_FLAT_LOW   = -40                   # Typelevel：-40 <= LevelArea < 50 → 平，其餘右下
FLAG_STAT        = dict(spike=8, strong=2, strong_days_gt=2)   # 沒有單日 > 8%，且 > 2% 的天數 > 2
VOL_WINDOW       = 3                     # 量能 / 漲跌 的滾動天數
VPC_MA_PCT_AS_PERCENT = False            # False = 維持舊版（比值，不乘 100）；True = 乘 100 變成 %

# ---------- 名單 ----------
EXCLUDED_TYPE1 = [
    '化學工業', '航運業', '生技醫療業', '塑膠工業', '金融保險', '鋼鐵工業', '居家生活',
]

HOLDINGS = [
    '1101', '1599', '1519', '2062', '2303', '2330', '2449', '6285', '6805', '8069',
]

# x 流：IG 推 https://www.instagram.com/moneyinallday/
WATCH_X = [
    '1519', '2308', '2342', '2357', '2379', '2383', '2401', '2436', '2454', '2465', '2498', '2723',
    '3006', '3035', '3138', '3169', '3217', '3231', '3374', '3483', '3526', '3552', '3661', '3708',
    '3715', '4927', '4968', '4976', '5243', '5425', '5457', '5474', '6223', '6231', '6271', '6278',
    '6438', '6441', '6515', '6525', '6533', '6679', '6689', '6789', '8069',
]

# y 流：創高 http://stock.tendy.net/tdin_profit_rf.php?open=1
WATCH_Y = [
    '2408', '6560', '3030', '2241', '8086', '6486', '3530', '5284', '6841', '3006', '8046', '4771',
    '6274', '1437', '3380', '6235', '6584', '6438', '8432', '3004', '2455', '1560', '2419', '5490',
    '8109', '1590', '4161', '3168', '6690', '2415', '6112', '5315', '2637', '3563', '4128', '1731',
    '6491', '4588', '6158', '4968', '3564', '4420', '2402', '6712', '3141', '6844', '4153',
]

# 要輸出成「分類資料夾」的標籤（其餘標籤只打標籤、不輸出）
EXPORT_FOLDERS = {
    '價_量':        'checksun_a',
    '【b1】':       'checksun_b1',
    '【b2】':       'checksun_b2',
    '【v】測試條件': 'checksun_v',
    # '【c】': 'checksun_c', '【d1】': 'checksun_d1', '【d2】': 'checksun_d2',
    # '【e】': 'checksun_e', '【x】看好': 'checksun_x', '【y】創高': 'checksun_y',
}

# 匯出時要固定成「小數 2 位」字串的欄位（其餘欄位直接轉字串）
FMT2_COLS = [
    'MA5_%', '均價_%', '均價longlong_%', 'MACDSL_%', 'MACD_%', 'MA_last_cross_%', 'highlight_%',
    'MACD', 'MACD-SL', '量能', '價能', '淨值倍率',
]

# fundamental_score_all 的型態標籤（不計入 score），併入 checkall 給 index.html 用（DailyFiles 已有「分類」欄，所以叫「基本面分類」）
FS_TYPE_COLS = ['基本面分類', 'Type_存股', 'Type_轉機', '轉機說明', 'Type_加速', 'Type_毛利結構',
                '景氣循環股', 'Type_景氣位置', '毛利率_歷史百分位', 'Type_警訊', '警訊說明']

# ---------- 輸出欄位（Excel 欄位順序就是這份清單的順序） ----------
OUTPUT_COLS = [
            '分類標籤',
            'stock_number','_quote' ,'成交股數','rt_量',
            'now_price', 'rt_price','diff_quote',

            '量倍率','VPC_MA_%',

            'highlight_date', 'LevelArea',

            'highlight_收盤價','highlight_%',

            'HH', 'HL', 'MA_UP', 'VOL_UP','TrendScore', 'TrendScore_15','TrendScore_30','TrendType', 'TrendScorertxt',
             'BB_MID', 'BB_UPPER', 'BB_LOWER', 'BB_WIDTH',
             'BB_MID_斜率',    'BB_UPPER_斜率', 'BB_WIDTH_斜率','BB_WIDTH_avg', 'BB_WIDTH_RATIO','BB_last_cross_date',
            'BB_golden_cross_area','BB_UPPER_pct_long', 'BB_UPPER_pct_short',
            'BB_UPPER_cross', 'BB_UPPER_cross_date',

            'up', 'up_last_cross_date',

             '綜合Flag', '綜合Flag_1d', '綜合Flag_2d', '綜合Flag_3d',
            #'my_signal', 'my_signal_3d','my_signal_5d','my_signal_10d',
            'up_up', 'up_up_3d', 'up_up_5d', 'up_up_10d',
            'bbw_cross',    'bbw_cross_3d',  'bbw_cross_5d',


            'MA_UP_d','MA_UP_3d', 'MA_UP_5d', 'MA_UP_10d', 'MA_UP_20d', 'MA_UP_30d',  'MA_UP_40d',
            'MarketState','MarketState_prev',

            'score', *FS_TYPE_COLS,


            '外資_累計買賣超', '投信_累計買進',
            '投信_累計賣出', '投信_累計買賣超', '自營商_累計買賣超', '三大法人_累計買賣超', '交易日數', '外資連續天數',
            '投信連續天數',
            'MA_death_cross','MA_break','MACD_bottom_signal',


            'MACD_hist_diff',
            'MA_death_cross_d', 'MA_death_cross_d收盤價', 'MA_death_cross_d2','MA_death_cross_d2收盤價',
            'MACD_death_cross_d', 'MACD_death_cross_d收盤價','MACD_death_cross_d2', 'MACD_death_cross_d2收盤價',
            'voc_cross', 'voc_cross_d',

            '日期','量能','價能','cond_entry',

            'text_desc',
        '趨勢多頭', '量起', '量起_K低', '價起', '價起_突破線', '價起回測', '量起後量縮守穩', '推薦進場',
        '推薦進場_類型' ,'量起_近5日', '推薦進場_近5日' , '量起_天數', '量起_狀態',

            '盤後量','成交量',
            '%K', '%D',
            'MA5_%', '均價_%','均價long_%' ,'均價longlong_%',
            'MA_5','MA_10','MA_20','MA_50', 'MA_80','MA_240',
            'MA_5_斜率', 'MA_10_斜率', 'MA_20_斜率', 'MA_50_斜率','MA_80_斜率','MA_240_斜率',
            'MACD_%', 'MACD','MACD-SL','MACD-SL_max','MACDSL_%',
            'MACD_last_cross_date',
            '成交金額平均','成交金額',

            #'Low_15d','Low_15d_date', 'Low_30d', 'Low_30d_date', 'Low_60d','Low_60d_date', 'Low_90d', 'Low_90d_date',
           #'High_15d','High_30d','High_60d','High_90d',

            'Volume_MA_5','Volume_MA_10','Volume_MA_30','Volume_MA_50','Volume_MA_80','volume_threshold',
             'VPC_MACD','成交金額_補值',

            'highlight_enddate','MA_last_cross_date收盤價','MA_last_cross_date','MA_last_cross_%',
            'stock_name',

            'Type0', 'Type1', 'Type2', '貝他值',
            '營業收入-上月比較增減(%)', '營業收入-去年同月增減(%)','累計營業收入-前期比較增減(%)',
            'Full_Summary','textdesc',

            '淨值倍率','殖利率','每股營收(元)','本益比','營業毛利率','營業利益率',
            '同業平均本益比','總市值','營收比重','Type','Typelevel',
            '開盤價', '最高價', '最低價', '收盤價','最小收盤價',
            '每股淨值(元)','desc','flag_stat',
            '盤整震盪_前15日次數','多頭排列_前5日次數','下跌帶量_前15日次數',
            'MA_break_前15日次數', '抄底訊號_前15日次數','黃金交叉_前15日次數','量起點_前15日次數',
            '價過5日_前15日次數','量短5長30比',
            'checkpoint_前10日次數', 'checkpoint_前5日次數','checkpoint_前2日次數','checkpoint_前1日次數',


             '季別', '營業收入', '營業成本', '營業毛利', '毛利率', '營業利益', '營益率',
             '業外收支', '稅前淨利', '稅後淨利', 'EPS(元)','EPS_前8季全為正','去年同季比較',
             'EPS_MA_8', 'GM_MA_8', 'EPS_維持','毛利率_維持', 'EPS_創新高', '毛利率_創新高', '財報健康', '財報強勢',
             'EPS_QoQ', 'EPS_QoQ_%', 'GM_QoQ', 'GM_QoQ_%', 'EPS_YoY','EPS_YoY_%', 'GM_YoY', 'GM_YoY_%', 'EPS_成長', 'EPS_YoY成長', '毛利率成長','毛利率YoY成長', '財報動能強',
           ]
OUTPUT_SET = set(OUTPUT_COLS)

# 匯出時額外加上的「族群」欄位（來自 supply_chain.json），插在 stock_number 後面
#   族群   = sector，例：半導體、AI伺服器（一檔可屬多個族群，用「、」分隔）
#   子族群 = sector_subsector，例：半導體_IC設計、AI伺服器_代工
SECTOR_COLS = ['族群', '子族群']
_i = OUTPUT_COLS.index('stock_number') + 1
EXPORT_COLS = OUTPUT_COLS[:_i] + SECTOR_COLS + OUTPUT_COLS[_i:]

## 2. 共用工具

In [ ]:
# =============================================================
# 2. 共用工具
# =============================================================
def to_float(series):
    """任何欄位 -> 數字（去掉 % 與千分位，無法轉換 -> NaN；True/False 也是 NaN）。
    數值儲存格直接轉 float、不經過字串：astype(str) 只留約 15 位有效數字，
    會讓 92.06500000000001 變成 '92.065'，四捨五入就從 92.07 跑成 92.06。"""
    if pd.api.types.is_numeric_dtype(series) and not pd.api.types.is_bool_dtype(series):
        return series.astype(float)
    is_num = series.map(lambda v: isinstance(v, (int, float, np.number)) and not isinstance(v, (bool, np.bool_)))
    txt = pd.to_numeric(
        series.astype(str)
              .str.replace('%', '', regex=False)
              .str.replace(',', '', regex=False)
              .str.strip(),
        errors='coerce',
    )
    return series.where(is_num, txt).astype(float)


def to_num(df, cols, fill=None, nd=None):
    """多個欄位一次轉數值（to_float），可選：缺值補 fill、四捨五入到 nd 位。原地修改並回傳 df。"""
    for c in ([cols] if isinstance(cols, str) else cols):
        s = to_float(df[c])
        if fill is not None:
            s = s.fillna(fill)
        if nd is not None:
            s = s.round(nd)
        df[c] = s
    return df


def strip_text(series, chars):
    """只處理字串儲存格，移除指定字元；非字串（含 NaN）原樣保留。"""
    def _f(v):
        if isinstance(v, str):
            for ch in chars:
                v = v.replace(ch, '')
        return v
    return series.map(_f)


# 新欄位 : (分子, 基準欄, 倍率)    公式 = (分子 - 基準) / |基準| * 倍率，四捨五入到小數 2 位
PCT_SPECS = {
    'highlight_%':     ('收盤價',      'highlight_收盤價',          100),
    'MA5_%':           ('收盤價',      'MA_5',                      100),
    '均價_%':          ('MA_5',        'MA_20',                     100),
    '均價long_%':      ('MA_20',       'MA_50',                     100),
    '均價longlong_%':  ('MA_50',       'MA_80',                     100),
    'MACDSL_%':        ('MACD-SL',     'MACD-SL_max',               100),
    'MACD_%':          ('MACD',        'MACD-SL',                   100),
    'VPC_MA_%':        ('Volume_MA_5', 'Volume_MA_10',              100 if VPC_MA_PCT_AS_PERCENT else 1),
    'MA_last_cross_%': ('now_price',   'MA_last_cross_date收盤價',  100),
}


def add_pct(df, specs=PCT_SPECS):
    """依 PCT_SPECS 一次算完所有乖離率；結果維持數值。基準為 0 時沿用舊行為（inf / NaN）。"""
    for name, (a, b, scale) in specs.items():
        x, y = to_float(df[a]), to_float(df[b])
        df[name] = ((x - y) / y.abs() * scale).round(2) + 0.0     # + 0.0：把 -0.0 變 0.0，避免匯出成 '-0.00'
    return df


def num(df, col):
    """規則判斷用：欄位 -> 數字。
    OUTPUT_COLS 內的欄位，NaN 視為 0（與舊版「輸出前 fillna('0')」一致）。"""
    s = to_float(df[col])
    return s.fillna(0) if col in OUTPUT_SET else s


def sanitize_filename(name):
    return re.sub(r'[\\/:*?"<>|]', '_', str(name))

## 3. 日期

In [3]:
# =============================================================
# 3. 日期
# =============================================================
def resolve_dates(now=None, override=TARGET_DATE_OVERRIDE, n_days=N_DAYS):
    """回傳 (adate, targetday, date_list)。
    週六 -> 週五、週日 -> 週五；平日休市（沒有即時檔）自動往前找最近有資料的日期。指定日期補跑請用 override。"""
    if override:
        target = datetime.strptime(override, '%Y-%m-%d').date()
    else:
        now = now or datetime.now()
        target = (now - timedelta(days={5: 1, 6: 2}.get(now.weekday(), 0))).date()
    fmt = '%Y-%m-%d'
    if not override:
        # 平日休市（國定假日、颱風假）沒有即時檔 → 改用最近一個有即時檔的日期
        avail = sorted(p.stem for p in REALTIME_DIR.glob('????-??-??.xlsx') if p.stem <= target.strftime(fmt))
        if avail and avail[-1] != target.strftime(fmt):
            print(f'⚠️ {target} 沒有即時檔（休市？），改用最近有資料的 {avail[-1]}')
            target = datetime.strptime(avail[-1], fmt).date()
    date_list = [(target - timedelta(days=i)).strftime(fmt) for i in range(n_days - 1, -1, -1)]
    return target.strftime(fmt), target.strftime(fmt), date_list


adate, targetday, date_list = resolve_dates()
print('目標日 targetday :', targetday)
print('Base 資料日期    :', date_list)

目標日 targetday : 2026-10-22
Base 資料日期    : ['2026-10-08', '2026-10-09', '2026-10-10', '2026-10-11', '2026-10-12', '2026-10-13', '2026-10-14', '2026-10-15', '2026-10-16', '2026-10-17', '2026-10-18', '2026-10-19', '2026-10-20', '2026-10-21', '2026-10-22']


## 4. 載入資料（函式定義）

In [4]:
# =============================================================
# 4. 載入資料
# =============================================================
def read_excels(paths):
    """逐檔讀取；失敗的檔案只警告、不中斷。"""
    frames = []
    for p in paths:
        try:
            frames.append(pd.read_excel(p))
        except Exception as e:
            print(f'⚠️ 讀取失敗 {p}：{e}')
    return frames


def load_daily_and_realtime(date_list, adate):
    """每日檔（多天）與即時檔 outer merge。
    沿用舊版：每日檔依「新→舊」排列、即時檔只保留 now_price 有值者、merge 用預設 suffix（_x/_y）。"""
    daily_frames = read_excels([DAILY_DIR / f'{d}.xlsx' for d in date_list])
    rt_frames = read_excels([REALTIME_DIR / f'{adate}.xlsx'])
    if not daily_frames:
        raise RuntimeError('找不到任何每日檔，請確認 DAILY_DIR 與日期')
    if not rt_frames:
        raise RuntimeError(f'找不到即時檔 {REALTIME_DIR / (adate + ".xlsx")}')
    daily = pd.concat(daily_frames[::-1])
    rt = pd.concat(rt_frames)
    rt = rt[pd.notna(rt['now_price'])]
    return pd.merge(daily, rt, on='stock_number', how='outer')


def load_supply_chain(path=SUPPLY_CHAIN_JSON):
    """供應鏈 JSON -> (detail_df 明細, detail_grouped 每檔合併成一段文字)。"""
    with open(path, 'r', encoding='utf-8') as f:
        detail_df = pd.DataFrame(json.load(f))
    detail_df['number'] = detail_df['number'].astype(str)
    detail_df = detail_df.rename(columns={'number': 'stock_number'})
    # 半導體｜上游｜IP設計/IC設計代工服務｜備註
    detail_df['text_desc'] = (detail_df[['wave', 'sector', 'subsector', 'memo']]
                              .astype(str).agg('｜'.join, axis=1))
    detail_grouped = (detail_df.groupby('stock_number', as_index=False)
                               .agg(text_desc=('text_desc', lambda x: '\r\n'.join(sorted(set(x))))))
    return detail_df, detail_grouped


def build_sector_map(detail_df):
    """每檔股票 -> 族群 / 子族群（多個以「、」串起來，依 supply_chain.json 出現順序）。"""
    d = detail_df.dropna(subset=['sector']).copy()
    d = d[d['sector'].astype(str) != '']
    d['_sd'] = d['sector'].astype(str) + '_' + d['subsector'].astype(str)
    join = lambda x: '、'.join(dict.fromkeys(x))
    return d.groupby('stock_number').agg(族群=('sector', join), 子族群=('_sd', join))


def load_stock_info(adate, detail_grouped):
    """基本面 / 財報 / 基金 / 評分等 Excel 全部併成一張 stock_info（一檔一列）。"""
    def rx(*parts):
        return pd.read_excel(RESULT_DIR.joinpath(*parts), engine='openpyxl')

    info = rx('StockInfo.xlsx')
    info['stock_number'] = info['stock_number'].astype(str)
    for name in ('StockProfitability.xlsx', 'StockType.xlsx'):
        part = rx(name)
        part['stock_number'] = part['stock_number'].astype(str)
        info = info.merge(part, on='stock_number', how='left')

    # 三大法人改由 attach_fund_daily 依日期併入，這裡不再併今天的 fund_result
    # score / 型態改由 attach_fund_score 直接併到 aa（金控等不在 StockInfo 的股票也拿得到）

    info = info.merge(detail_grouped, on='stock_number', how='left')

    # 文字欄位清理
    info['desc'] = info['desc'].fillna('-')
    for c in ('成交量', '總市值', '一年內最大量', '一年內最低量'):
        info[c] = strip_text(info[c], ',')
    info['殖利率'] = strip_text(info['殖利率'], '%')
    for c in ('營業毛利率', '營業利益率'):
        info[c] = strip_text(info[c], '%')       # 舊版漏了 .str，% 其實沒被去掉
    return info


def attach_fund_score(aa):
    """併入 fundamental_score_all 的 score 與型態欄位。
    直接依 stock_number 併到 aa，不經 StockInfo：金控、部分新上市不在 StockInfo，以前 score 會變 0。"""
    score = pd.read_excel(RESULT_DIR / 'fundamental_score_all.xlsx', engine='openpyxl')
    score['stock_number'] = score['stock_number'].astype(str)
    # reindex：舊版 fundamental_score_all 沒有型態欄位時補空值，不讓 build_export_frame 報缺欄
    score = score.reindex(columns=['stock_number', 'score', *FS_TYPE_COLS])
    aa = aa.merge(score, on='stock_number', how='left')
    aa['基本面分類'] = aa['基本面分類'].fillna('0 無財報資料')   # 不在評分檔 = 沒有財報資料
    return aa


FUND_COLS = ['外資_累計買賣超', '投信_累計買進', '投信_累計賣出', '投信_累計買賣超', '自營商_累計買賣超',
             '三大法人_累計買賣超', '交易日數', '外資連續天數', '投信連續天數']


def attach_fund_daily(aa, date_list):
    """三大法人依 (stock_number, 日期) 併入：每一天用「當天」的 fund_result（單位：股）。
    舊版只讀今天的檔、依代號併，過去日期的列都拿到今天的法人資料。"""
    frames = []
    for d in date_list:
        p = RESULT_DIR / 'fund' / f'fund_result_{d.replace("-", "")}.xlsx'
        if not p.exists():
            continue
        f = pd.read_excel(p, engine='openpyxl')
        f['stock_number'] = f['code'].astype(str).str.strip()
        f['日期'] = d
        frames.append(f[['stock_number', '日期'] + FUND_COLS].drop_duplicates('stock_number'))
    fund = (pd.concat(frames, ignore_index=True) if frames
            else pd.DataFrame(columns=['stock_number', '日期'] + FUND_COLS))
    missing = sorted(set(aa['日期'].dropna()) - set(fund['日期']))
    if missing:
        print('⚠️ 這些日期沒有 fund_result，法人欄位為空：', missing)
    return aa.merge(fund, on=['stock_number', '日期'], how='left')

## 5. 特徵計算（函式定義）

In [ ]:
# =============================================================
# 5. 特徵計算（全程維持數值；只有匯出時才轉字串）
# =============================================================
RENAME_MERGED = {'日期_x': '日期', 'now_price_x': 'now_price',
                 'change_price_x': 'change_price', 'change_quote_x': 'change_quote'}
RENAME_AFTER_INFO = {'收盤價_x': '收盤價', '最低價_x': '最低價', '最高價_x': '最高價', '開盤價_x': '開盤價'}


def attach_stock_base(df, base_csv=BASE_INFO_CSV):
    """一次 map 出 名稱 / 產業 / 市場（取代舊版逐列 apply 的 GetStockInfoByID）。"""
    base = (pd.read_csv(base_csv, dtype={'公司代號': str})
              .drop_duplicates('公司代號').set_index('公司代號'))
    base['_category'] = base['上櫃公司產業類別'].where(base['Type'] == '上櫃', base['上市公司產業類別'])
    sid = df['stock_number'].astype(str)
    known = sid.isin(base.index)

    def lookup(col):                      # 找不到 -> 'None'（與舊版相同）
        return sid.map(base[col]).astype(object).where(known, 'None').map(str)

    df['Type0'], df['Type1'], df['Type2'] = lookup('公司簡稱'), lookup('_category'), lookup('Type')
    df['Type'] = df['Type0'] + '-' + df['Type1'] + '-' + df['Type2']
    df['stock_name'] = df['Type0']
    return df


def transform_daily(df):
    """合併後的原始表 -> 加上各種指標。"""
    df = df.rename(columns=RENAME_MERGED)
    for col in ('日期', 'highlight_date', 'highlight_enddate'):
        df[col] = pd.to_datetime(df[col], errors='coerce').dt.strftime('%Y-%m-%d')
    df['stock_number'] = df['stock_number'].astype(str)
    df = attach_stock_base(df)

    add_pct(df)                                                       # 所有乖離率 / 比值（PCT_SPECS）

    # --- 數值整理（乖離率算完之後才補 0 / 取位，避免影響上面的計算）---
    to_num(df, ['MACD', 'MACD-SL'], nd=2)
    to_num(df, ['now_price', '收盤價', '漲跌價差'], fill=0)
    to_num(df, ['開盤價', '成交股數', 'VPC_MACD'])
    df['level'] = to_float(df['level']) * 100
    for c in ('MA_break', 'MACD_golden_cross'):
        df[c] = df[c].fillna(False)

    df = df.sort_values(['stock_number', '日期']).reset_index(drop=True)
    df['前日MACD-SL'] = df.groupby('stock_number')['MACD-SL'].shift(1)   # SS 分流用

    # 當日漲跌幅 _quote = 漲跌價差 / 開盤價 * 100（開盤價 0 -> 0）
    open_ = df['開盤價']
    df['_quote'] = (df['漲跌價差'] / open_ * 100).round(2).where(open_ != 0, 0.0)

    # LevelArea：負區間取負號
    df['LevelArea'] = df['level'] * np.where(df['interval_type'] == '負區間', -1, 1)

    # 上市成交股數換算成「張」
    shares = df['成交股數']
    df['成交股數'] = shares.where(df['Type2'] != '上市', shares / 1000)

    # Typelevel：族群 + 右上 / 平 / 右下 + 市場別
    la = df['LevelArea']
    zone = pd.Series(np.select([la >= LEVEL_UP, (la >= LEVEL_FLAT_LOW) & (la < LEVEL_UP)],
                               ['右上', '平'], default='右下'), index=df.index)
    df['Typelevel'] = df['Type1'] + zone + df['Type2']

    df = df.replace('-', 0)

    # flag_stat：一段期間內沒有暴衝(>8%)、但有 3 天以上漲幅 > 2%
    df['_quote'] = df['_quote'].fillna(0)
    by = df['stock_number']
    no_spike    = (df['_quote'] > FLAG_STAT['spike']).groupby(by).transform('sum') == 0
    strong_days = (df['_quote'] > FLAG_STAT['strong']).groupby(by).transform('sum') > FLAG_STAT['strong_days_gt']
    df['flag_stat'] = no_spike & strong_days

    # 其他欄位統一轉數值、補 0、取 2 位
    to_num(df, ['收盤價', '最低價', '最高價', '%K', '%D', 'MACD-SL_max', 'MA_20', 'MA_50', 'MA_80',
                'Volume_MA_5', 'Volume_MA_50', '均價long_%', 'MACD_hist_diff'], fill=0, nd=2)
    return df


def enrich_with_info(df, info):
    """併入基本面表，並算出 量能 / 淨值倍率 / diff_quote / 價能。"""
    aa = df.merge(info, on='stock_number', how='left').copy()         # 預設 suffix：與舊版相同；copy 以整理欄位碎片

    aa['營收比重'] = aa['營收比重'].fillna('-')
    to_num(aa, ['now_price', 'now_price_y', '每股淨值(元)', '本益比', '一年內最大量'])
    to_num(aa, '成交量', fill=0)
    aa['成交量'] = aa['成交量'].astype('int64')

    aa['量能']     = (aa['成交股數'] / aa['一年內最大量'].fillna(0) * 100).round(2)   # 今日量 / 一年內最大量 (%)
    aa['淨值倍率'] = (aa['now_price'] / aa['每股淨值(元)']).round(2)

    y = aa['now_price_y']                                             # 即時價 vs 昨收 的差距 (%)
    aa['diff_quote'] = ((y - aa['收盤價_x']) / y * 100).round(2).where(y != 0)

    aa = aa.rename(columns=RENAME_AFTER_INFO)

    # 價能：收盤價靠近高點(+) / 低點(-)
    aa['價能'] = (((aa['收盤價'] - aa['最低價']) / aa['最低價']
                  - (aa['最高價'] - aa['收盤價']) / aa['收盤價']) * 100).round(2)
    aa['rt_price'] = ''
    aa['rt_量'] = ''
    return aa


def _volume_features_one(g, n):
    """單一股票：增縮 / 漲跌 的 N 日滾動、textdesc、量倍率。"""
    g = g.sort_values('日期_').reset_index(drop=True)
    vol = to_float(g['量能']).replace([np.inf, -np.inf], np.nan)     # inf 會讓 astype(int) 爆掉，先當缺值

    g['增縮BYN天']      = vol.rolling(n).sum().fillna(0).astype(int)
    g['增縮BYN天Diff']  = g['增縮BYN天'].diff()
    g['漲跌BYN天']      = g['_quote'].rolling(n).sum().fillna(0).astype(int)
    g['漲跌BYN天Diff']  = g['漲跌BYN天'].diff()

    p, v = g['漲跌BYN天Diff'], g['增縮BYN天Diff']
    price_desc  = np.select([p > 2, p < -3],  ['漲', '跌'], default='-')
    volume_desc = np.select([v > 2, v < -10], ['增', '縮'], default='-')
    g['textdesc'] = ['價' + a + '量' + b for a, b in zip(price_desc, volume_desc)]

    # --- 量倍率 = Σ(漲>3% 的近 7 筆 量比) - Σ(跌<-2% 的近 7 筆 量比)，量比 = 量能 / 量能均 ---
    vol_ma = to_float(g['量能均']) if '量能均' in g.columns else vol.rolling(n, min_periods=1).mean()
    g_desc = g.sort_values('日期_', ascending=False)
    vol_d, ma_d = vol.reindex(g_desc.index), vol_ma.reindex(g_desc.index)
    ratio = np.round(np.where(ma_d > 0, vol_d / ma_d, np.nan), 2)
    # ⚠️ 照舊版寫法：pd.Series(ratio) 的 index 是 0..n-1（新→舊的順位），
    #    但 mask 的 index 是原始列號，pandas 會依 index 對齊 → ratio 與 mask 實際上錯位。
    #    為了不改變既有數值先照舊；若要改成「真的取最近 7 筆」，把下一行改成
    #    ratio = pd.Series(ratio, index=g_desc.index)
    ratio = pd.Series(ratio)
    add   = ratio[g_desc['_quote'] > 3].head(7).fillna(0).sum()
    minus = ratio[g_desc['_quote'] < -2].head(7).fillna(0).sum()
    g['量倍率'] = float(np.round(add - minus, 2))
    return g


def add_volume_features(df, n=VOL_WINDOW):
    df = df.copy()
    df['日期_'] = pd.to_datetime(df['日期'])
    parts = [_volume_features_one(g, n) for _, g in df.groupby('stock_number', sort=True)]
    return pd.concat(parts).reset_index(drop=True)


def add_volume_watch(df, max_age=10):
    """近日量起 觀察：最近一次量起到今天第幾個交易日（量起_天數），以及目前走到哪一步（量起_狀態）。
    狀態：量起(D+0) → 等量縮(D+1~2) → 等守穩(D+3~10，等「量起後量縮守穩」) → ★推薦；
          量起後最低價跌破量起K低 → 失效。超過 max_age 個交易日就不顯示。
    只看 date_list 內的資料（約 10 個交易日），更早的量起看不到。
    回測（2015~2026）：量起當天追價無超額；等到推薦（趨勢多頭 + 量縮守穩）20 日超額約 +1.9%。"""
    is_true = lambda s: s.astype(str).str.lower().eq('true')
    ages, states = {}, {}
    d = df[df['日期_'].notna()].sort_values(['stock_number', '日期_'])
    for _, g in d.groupby('stock_number', sort=False):
        vol = is_true(g['量起']).to_numpy()
        if not vol.any():
            continue
        rec  = is_true(g['推薦進場']).to_numpy()
        low  = to_float(g['最低價']).to_numpy()
        klow = to_float(g['量起_K低']).to_numpy()
        last = -1
        for k, ix in enumerate(g.index):
            if vol[k]:
                last = k
            age = k - last
            if last < 0 or age > max_age:
                continue
            after = slice(last + 1, k + 1)
            if rec[after].any():
                st = '★推薦'
            elif age > 0 and np.nanmin(low[after]) < klow[last]:
                st = '失效'
            else:
                st = '量起' if age == 0 else '等量縮' if age <= 2 else '等守穩'
            ages[ix], states[ix] = age, st
    df = df.copy()
    df['量起_天數'] = pd.Series(ages, dtype=float)
    df['量起_狀態'] = pd.Series(states, dtype=object).reindex(df.index).fillna('')
    return df


### 5-1. 執行：載入 → 特徵

In [6]:
# 執行：載入 -> 特徵
raw = load_daily_and_realtime(date_list, adate)
df = transform_daily(raw)
detail_df, detail_grouped = load_supply_chain()
stock_info = load_stock_info(adate, detail_grouped)
aa = enrich_with_info(df, stock_info)
aa = attach_fund_score(aa)
aa = attach_fund_daily(aa, date_list)
aa = add_volume_features(aa)
aa = add_volume_watch(aa)

sector_map = build_sector_map(detail_df)
for col in SECTOR_COLS:
    aa[col] = aa['stock_number'].map(sector_map[col]).fillna('-')
_d = aa['日期'].dropna()      # 只在即時檔出現的股票，日期是空值，要先排除才能取 min/max
print(f'資料筆數 {len(aa):,}，股票 {aa["stock_number"].nunique():,} 檔，日期 {_d.min()} ~ {_d.max()}，'
      f'日期缺值 {aa["日期"].isna().sum()} 列')


def peek(stock, day=None, cols=None):
    """除錯用：看某檔(某天)的資料，取代舊版散落各處的 df[...] 檢查 cell。"""
    m = aa['stock_number'] == str(stock)
    if day:
        m &= aa['日期'] == day
    return aa.loc[m, cols] if cols else aa[m]

⚠️ 讀取失敗 D:\Project\Jupyter\Stock\Main\Result\DailyFiles\2026-10-09.xlsx：[Errno 2] No such file or directory: 'D:\\Project\\Jupyter\\Stock\\Main\\Result\\DailyFiles\\2026-10-09.xlsx'
⚠️ 讀取失敗 D:\Project\Jupyter\Stock\Main\Result\DailyFiles\2026-10-10.xlsx：[Errno 2] No such file or directory: 'D:\\Project\\Jupyter\\Stock\\Main\\Result\\DailyFiles\\2026-10-10.xlsx'
⚠️ 讀取失敗 D:\Project\Jupyter\Stock\Main\Result\DailyFiles\2026-10-11.xlsx：[Errno 2] No such file or directory: 'D:\\Project\\Jupyter\\Stock\\Main\\Result\\DailyFiles\\2026-10-11.xlsx'
⚠️ 讀取失敗 D:\Project\Jupyter\Stock\Main\Result\DailyFiles\2026-10-17.xlsx：[Errno 2] No such file or directory: 'D:\\Project\\Jupyter\\Stock\\Main\\Result\\DailyFiles\\2026-10-17.xlsx'
⚠️ 讀取失敗 D:\Project\Jupyter\Stock\Main\Result\DailyFiles\2026-10-18.xlsx：[Errno 2] No such file or directory: 'D:\\Project\\Jupyter\\Stock\\Main\\Result\\DailyFiles\\2026-10-18.xlsx'
資料筆數 19,676，股票 1,975 檔，日期 2026-10-08 ~ 2026-10-22，日期缺值 0 列


## 6. 統計

In [ ]:
# =============================================================
# 6. 統計：Typelevel / 族群 的成交金額與強勢股數
# =============================================================
def group_stats(df, by, hot=HOT_QUOTE):
    """依 (日期, by) 彙總：成交金額(百萬)、檔數、強勢股數(漲幅 > hot)。"""
    d = df.assign(_amt=to_float(df['成交金額_補值']),
                  _hot=(to_float(df['_quote']) > hot).astype(int))
    return (d.groupby(['日期', by])
              .agg(amount=('_amt', 'sum'), count_total=('stock_number', 'count'), count_hot=('_hot', 'sum'))
              .assign(amount=lambda x: x['amount'] / 1_000_000)
              .reset_index())


def save_pivots(stats, by, hot_file, amount_file):
    stats.pivot(index=by, columns='日期', values='count_hot').to_excel(RESULT_DIR / hot_file)
    stats.pivot(index=by, columns='日期', values='amount').to_excel(RESULT_DIR / amount_file)


def top_n(stats, by, day, n):
    return stats.pivot(index=by, columns='日期', values='amount')[day].sort_values(ascending=False).head(n).index


RESULT_DIR.mkdir(parents=True, exist_ok=True)

# Typelevel
type_stats = group_stats(aa, 'Typelevel')
save_pivots(type_stats, 'Typelevel', 'analyze_1.xlsx', 'analyze_2.xlsx')
top5_index = top_n(type_stats, 'Typelevel', targetday, 5)
print('Typelevel 成交金額 Top5：', top5_index.tolist())

# 族群（供應鏈 sector）
sector_df = aa[['stock_number', '日期', '成交金額_補值', '成交金額', '_quote', '收盤價', 'stock_name']].merge(
    detail_df[['stock_number', 'sector', 'subsector']], on='stock_number', how='left')
sector_stats = group_stats(sector_df, 'sector')
save_pivots(sector_stats, 'sector', 'analyze_3.xlsx', 'analyze_4.xlsx')
top10_index = top_n(sector_stats, 'sector', targetday, 10)
print('族群 成交金額 Top10：', top10_index.tolist())

## 7. 選股規則

In [ ]:
# =============================================================
# 7. 選股規則：標籤 + 條件 + 輸出資料夾，全部集中在 RULES
#    新增條件 = 在 build_conditions 加一個條件，並在 RULES 加一行
# =============================================================
def build_conditions(aa, targetday):
    n = lambda col: num(aa, col)
    is_today = aa['日期'] == targetday

    avg_pct, long_pct, macd_pct = n('均價_%'), n('均價long_%'), n('MACD_%')
    vol5, vol10, vol30 = n('Volume_MA_5'), n('Volume_MA_10'), n('Volume_MA_30')

    # 判斷用的輔助欄位（【v】條件用；前端 index.html 會自己再算一次顯示用的值）
    aa['現量5d比'] = (n('成交金額') / vol5.replace(0, np.nan)).round(2)

    c = {}
    # A：價量
    c['價_量'] = (
        (~aa['Type1'].isin(EXCLUDED_TYPE1))
        & aa['Full_Summary'].astype(str).str.contains('多頭排列', na=False)
        & (n('成交股數') > 100)
        & (n('LevelArea') > 0)
        & ((n('Volume_MA_50') / n('成交金額平均').replace(0, np.nan)) > 0.5)
        & (n('營業利益率') > 0)
        & (n('累計營業收入-前期比較增減(%)') > 0)
        & (n('MACD_hist_diff') > 0)
        & (vol5  > n('Volume_MA_50') * 0.9)
        & (vol10 > n('Volume_MA_50') * 0.9)
    )
    # B ~ E：均價_% / 均價long_% / MACD_% 的正負組合
    c['【b1】'] = (avg_pct >= 0) & (long_pct > 0) & (macd_pct > 0)
    c['【b2】'] = (avg_pct <  0) & (long_pct > 0) & (macd_pct > 0)
    c['【c】']  =                  (long_pct > 0) & (macd_pct < 0)
    c['【d1】'] = (avg_pct >= 0) & (long_pct < 0) & (macd_pct > 0)
    c['【d2】'] = (avg_pct <  0) & (long_pct < 0) & (macd_pct > 0)
    c['【e】']  =                  (long_pct < 0) & (macd_pct < 0)
    # x / y 觀察名單（僅當日）
    c['【x】看好'] = is_today & aa['stock_number'].isin(WATCH_X)
    c['【y】創高'] = is_today & aa['stock_number'].isin(WATCH_Y)
    # V：量能遞增（僅當日）
    c['【v】測試條件'] = (is_today & (n('up_up_10d') > 0) & (vol5 > vol10) & (vol10 > vol30)
                      & (aa['現量5d比'] > 1) & (vol5 > 10000))
    return c, is_today


def stocks_where(aa, cond):
    return aa.loc[cond, 'stock_number'].drop_duplicates().tolist()


conds, is_today = build_conditions(aa, targetday)

# 打標籤（同一檔可同時符合多個條件，標籤用 ; 分隔）
tags = pd.Series('', index=aa.index)
for label, cond in conds.items():
    tags = tags + np.where(cond, label + ';', '')
aa['分類標籤'] = tags.str.strip(';')

print(aa.loc[is_today, '分類標籤'].str.split(';').explode().value_counts().drop('', errors='ignore'))

## 8. 匯出

In [9]:
# =============================================================
# 8. 匯出：只有在這裡才把欄位轉成字串
# =============================================================
def build_export_frame(aa):
    missing = [c for c in EXPORT_COLS if c not in aa.columns]
    if missing:
        raise KeyError(f'EXPORT_COLS 有 {len(missing)} 個欄位在資料中找不到：{missing}')
    out = aa[EXPORT_COLS].copy()
    for c in FMT2_COLS:
        if c in out.columns:
            out[c] = to_float(out[c]).map('{:.2f}'.format)
    out = out.astype(object).where(out.notna(), '0').astype(str)
    return out.replace(['NaN', 'nan', 'None', ''], '0')


def export_history(exp, stocks, folder, filename, only_date=None, verbose=True):
    """指定股票的完整歷史 -> Excel（自動建資料夾、去重、依代號/日期由新到舊排序）。"""
    folder = Path(folder)
    folder.mkdir(parents=True, exist_ok=True)
    mask = exp['stock_number'].isin([str(s) for s in stocks])
    if only_date is not None:
        mask &= exp['日期'] == only_date
    (exp[mask].drop_duplicates()
        .sort_values(by=['stock_number', '日期'], ascending=False)
        .to_excel(folder / filename, index=False))
    if verbose:
        print(f'{folder / filename}  {len(set(stocks))} 檔')


export_df = build_export_frame(aa)

# ---- 依標籤輸出 ----
for label, folder in EXPORT_FOLDERS.items():
    stocks = stocks_where(aa, conds[label] & is_today)
    export_history(export_df, stocks, RESULT_DIR / folder, f'{targetday}.xlsx')

# ---- SS 分流：近 6 日 MACD-SL 由負轉正，且當日 MACD_% >= 0 ----
cond_ss_cross = ((num(aa, 'MACD-SL') >= 0) & (num(aa, '前日MACD-SL') <= 0)
                 & aa['日期'].isin(date_list[-6:]))
cond_ss_today = (num(aa, 'MACD_%') >= 0) & is_today
export_history(export_df, set(stocks_where(aa, cond_ss_cross)) & set(stocks_where(aa, cond_ss_today)),
               RESULT_DIR / 'checksun_ss', f'{targetday}.xlsx')

# ---- 持股 / 各產業 ----
export_history(export_df, HOLDINGS, RESULT_DIR, 'checksunmy.xlsx')
for type1 in sorted(aa['Type1'].astype(str).unique()):
    export_history(export_df, stocks_where(aa, aa['Type1'] == type1),
                   RESULT_DIR / 'checksun', f'{sanitize_filename(type1)}.xlsx', verbose=False)

# ---- 全市場每日檔(checkall) / 持股當日(checkmy) / 量能遞增當日(check_v) ----
# 過去日期的檔是當天的快照（diff_quote、基本面都是當時的值），已存在就不重寫
for d in date_list:
    if d != targetday and (RESULT_DIR / 'checkall' / f'{d}.xlsx').exists() and not REBUILD_CHECKALL:
        continue
    day_stocks = aa.loc[aa['日期'] == d, 'stock_number'].unique().tolist()
    if not day_stocks:                    # 週末 / 假日沒資料，不產生空檔
        continue
    export_history(export_df, day_stocks, RESULT_DIR / 'checkall', f'{d}.xlsx', only_date=d, verbose=False)
export_history(export_df, HOLDINGS, RESULT_DIR, 'checkmy.xlsx', only_date=targetday)
export_history(export_df, stocks_where(aa, conds['【v】測試條件']),
               RESULT_DIR / 'check_v', f'{targetday}.xlsx', only_date=targetday)

D:\Project\Jupyter\Stock\Main\Result\checksun_a\2026-10-22.xlsx  13 檔
D:\Project\Jupyter\Stock\Main\Result\checksun_b1\2026-10-22.xlsx  244 檔
D:\Project\Jupyter\Stock\Main\Result\checksun_b2\2026-10-22.xlsx  10 檔
D:\Project\Jupyter\Stock\Main\Result\checksun_v\2026-10-22.xlsx  27 檔
D:\Project\Jupyter\Stock\Main\Result\checksun_ss\2026-10-22.xlsx  61 檔
D:\Project\Jupyter\Stock\Main\Result\checksunmy.xlsx  10 檔
D:\Project\Jupyter\Stock\Main\Result\checkmy.xlsx  10 檔
D:\Project\Jupyter\Stock\Main\Result\check_v\2026-10-22.xlsx  27 檔


## 9. 族群檔與 summary.json

In [11]:
# =============================================================
# 9-1. 依族群(sector)各輸出一個 Excel：Result/checksun_type/<族群>.xlsx
# =============================================================
type_dir = RESULT_DIR / 'checksun_type'
type_dir.mkdir(parents=True, exist_ok=True)
sector_members = detail_df.dropna(subset=['sector']).query("sector != ''").groupby('sector')['stock_number'].unique()

for sector, members in sorted(sector_members.items()):
    safe = sanitize_filename(sector)
    print(safe)
    # text_desc 清空：同一檔在多個子族群的列才會被 drop_duplicates 合併（與舊版一致）
    sub = (export_df[export_df['stock_number'].isin(members)]
           .assign(text_desc='0')
           .drop_duplicates()
           .sort_values(by=['stock_number', '日期'], ascending=False))
    sub.to_excel(type_dir / f'{safe}.xlsx', index=False)

AI伺服器
AI應用服務
AI晶片
AI電力基建
PCB載板
光學
光通訊
光電
其他
其他電子
功率元件
化學
半導體
半導體材料
半導體設備
封裝測試
導線架
居家生活
工業自動化
工業電腦
數位雲端
文化創意
機器人
橡膠
水泥
汽車
油電燃氣
營建
玻璃
玻璃陶瓷
生技
生技醫療
石化
紡織
綠能
網通
航空
航運
被動元件
觀光
記憶體
貿易百貨
資訊服務
軍工
農業科技
通信網路
造紙
連接器
運動休閒
醫療通路
金融
鋼鐵
零售
電信
電動車
電器電纜
電子紙
電子通路
電子零組件
電機機械
電腦週邊
面板
食品


In [ ]:
# =============================================================
# 9-2. 產出 Result/summary.json（族群儀表板用）
# =============================================================
def trend_change(vol, inst, surge=1.5):
    """族群訊號（儀表板卡片用）：量看熱度、法人看方向。
    breakout      今日量 > 前 5 日均量 × surge，且今日法人淨買
    accelerate    量連 3 日增加，且近 2 日法人皆淨買
    topping       今日爆量但法人淨賣，或量從 10 日高點縮到 7 成以下
    weak_continue 近 3 日法人皆淨賣，且量連 2 日減少"""
    if len(vol) < 6:
        return None
    avg5 = sum(vol[-6:-1]) / 5
    spike = avg5 > 0 and vol[-1] > avg5 * surge
    if spike and inst[-1] > 0:
        return 'breakout'
    if vol[-1] > vol[-2] > vol[-3] and inst[-1] > 0 and inst[-2] > 0:
        return 'accelerate'
    if (spike and inst[-1] < 0) or (vol[-2] == max(vol) and vol[-1] < vol[-2] * 0.7):
        return 'topping'
    if all(x < 0 for x in inst[-3:]) and vol[-1] < vol[-2] < vol[-3]:
        return 'weak_continue'
    return None


def build_summary(sector_df, day, hot=HOT_QUOTE, top_k=2):
    d = sector_df.copy()
    d['sectordetail'] = d['sector'] + '_' + d['subsector']
    to_num(d, ['成交金額_補值', '成交金額', '收盤價', '_quote', '三大法人_累計買賣超'])
    d['_inst'] = d['三大法人_累計買賣超'] * d['收盤價'] / 1e8          # 法人淨買賣金額（億）
    all_dates = sorted(d['日期'].dropna().unique().tolist())
    groups = dict(tuple(d.groupby('sectordetail')))

    summary, dates = {}, []
    for sector in d['sectordetail'].dropna().unique():
        sec = groups[sector]
        vol_s   = (sec.groupby('日期')['成交金額_補值'].sum() / 1_000_000).round(2)
        hot_s   = sec[sec['_quote'] > hot].groupby('日期')['stock_number'].count()
        total_s = sec.groupby('日期')['stock_number'].count()
        inst_s  = sec.groupby('日期')['_inst'].sum()

        dates = [x for x in all_dates if x in vol_s.index]
        top = sec[sec['日期'] == day].nlargest(top_k, '成交金額')
        vols  = [round(float(vol_s.get(x, 0)), 2) for x in dates]
        insts = [round(float(inst_s.get(x, 0)), 2) for x in dates]
        hots  = [int(hot_s.get(x, 0)) for x in dates]
        totals = [int(total_s.get(x, 0)) for x in dates]
        summary[str(sector)] = {
            'dates':       dates,
            'vol':         vols,
            'inst':        insts,
            'hot':         hots,
            'totals':      totals,
            'trend_change': trend_change(vols, insts),
            'hot_sector':  bool(hots and hots[-1] >= 2 and hots[-1] / max(totals[-1], 1) >= 0.2),
            'total':       int(sec['stock_number'].nunique()),
            'stock_count': int(sec['stock_number'].nunique()),
            'rep_stocks': [{
                'id':    str(r['stock_number']),
                'name':  str(r['stock_name']),
                'price': float(r['收盤價']) if pd.notna(r['收盤價']) else 0.0,
                'quote': float(r['_quote']) if pd.notna(r['_quote']) else 0.0,   # 沿用舊版：quote = _quote
                'vol':   float(r['成交金額']) if pd.notna(r['成交金額']) else 0.0,
            } for _, r in top.iterrows()],
        }
    return summary, dates


inst_df = aa[['stock_number', '日期', '三大法人_累計買賣超']].drop_duplicates(['stock_number', '日期'])
summary, dates = build_summary(sector_df.merge(inst_df, on=['stock_number', '日期'], how='left'), targetday)
with open(RESULT_DIR / 'summary.json', 'w', encoding='utf-8') as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)
print(f'✅ summary.json 輸出完成，{len(summary)} 個產業，日期：{dates}')